# Chapter 36: Autoencoders from Scratch

## Learning Objectives
- Build encoder-decoder with manual gradients
- Learn compact latent representation
- Use reconstruction error for anomaly detection
- Study bottleneck dimension tradeoffs

## Prerequisites
- Strong understanding of chapters 0-29
- Python and basic linear algebra
- Numpy for probabilistic/deep chapters


# Chapter 36: Autoencoders from Scratch (Representation Learning)

## Why this chapter matters
Autoencoders are a foundation for representation learning, anomaly detection, denoising, and pretraining workflows.

## Learning goals
1. Build a shallow autoencoder with manual backprop.
2. Understand bottleneck latent representations.
3. Use reconstruction error as anomaly signal.
4. Compare undercomplete vs overcomplete bottleneck settings.

## Model
Encoder:
\[
z = \sigma(XW_1 + b_1)
\]
Decoder:
\[
\hat{X} = zW_2 + b_2
\]
Loss:
\[
\mathcal{L} = \frac{1}{N}\sum_i ||x_i - \hat{x}_i||_2^2
\]

## Step-by-step training loop
1. Forward pass encoder + decoder.
2. Compute reconstruction loss.
3. Backprop gradients for both layers.
4. Update parameters.
5. Track reconstruction error per epoch.

## Industry use cases
- anomaly detection via reconstruction error threshold
- dimensionality reduction before clustering
- denoising corrupted sensor data

## Assignment
1. Train autoencoder on clean data and test on corrupted samples.
2. Plot reconstruction error distribution.
3. Set threshold based on validation quantile.


In [ ]:
from __future__ import annotations

import numpy as np


class Autoencoder:
    def __init__(self, in_dim: int, latent_dim: int, seed: int = 42):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 0.2, size=(in_dim, latent_dim))
        self.b1 = np.zeros((1, latent_dim))
        self.W2 = rng.normal(0, 0.2, size=(latent_dim, in_dim))
        self.b2 = np.zeros((1, in_dim))

    @staticmethod
    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-x))

    def forward(self, X):
        h_pre = X @ self.W1 + self.b1
        h = self.sigmoid(h_pre)
        X_hat = h @ self.W2 + self.b2
        cache = (X, h_pre, h, X_hat)
        return X_hat, cache

    @staticmethod
    def mse_loss(X, X_hat):
        return np.mean((X - X_hat) ** 2)

    def backward(self, cache):
        X, h_pre, h, X_hat = cache
        n = X.shape[0]

        dX_hat = 2.0 * (X_hat - X) / n
        dW2 = h.T @ dX_hat
        db2 = np.sum(dX_hat, axis=0, keepdims=True)

        dh = dX_hat @ self.W2.T
        dh_pre = dh * h * (1 - h)
        dW1 = X.T @ dh_pre
        db1 = np.sum(dh_pre, axis=0, keepdims=True)

        return dW1, db1, dW2, db2

    def step(self, grads, lr=0.05, weight_decay=1e-4):
        dW1, db1, dW2, db2 = grads
        self.W1 -= lr * (dW1 + weight_decay * self.W1)
        self.b1 -= lr * db1
        self.W2 -= lr * (dW2 + weight_decay * self.W2)
        self.b2 -= lr * db2


def make_data(n=300, seed=0):
    rng = np.random.default_rng(seed)
    x1 = rng.normal(0, 1, size=(n, 1))
    x2 = 0.7 * x1 + rng.normal(0, 0.2, size=(n, 1))
    x3 = -0.4 * x1 + 0.5 * x2 + rng.normal(0, 0.2, size=(n, 1))
    X = np.hstack([x1, x2, x3])
    return X


if __name__ == "__main__":
    X = make_data(400)
    model = Autoencoder(in_dim=3, latent_dim=2)

    for epoch in range(1, 501):
        X_hat, cache = model.forward(X)
        loss = model.mse_loss(X, X_hat)
        grads = model.backward(cache)
        model.step(grads, lr=0.05)

        if epoch % 50 == 0:
            print(f"epoch={epoch:03d} recon_loss={loss:.6f}")

    # anomaly score example
    X_noisy = X.copy()
    X_noisy[:10] += 2.5
    X_hat_noisy, _ = model.forward(X_noisy)
    errors = np.mean((X_noisy - X_hat_noisy) ** 2, axis=1)
    print("Top anomaly scores:", np.round(np.sort(errors)[-10:], 4))


## Checkpoint

1. You can derive reconstruction MSE gradients.
2. You can train an undercomplete autoencoder.
3. You can score anomalies via reconstruction error.

## Guided Exercise
Test `latent_dim` values and compare final reconstruction loss and anomaly separation.

In [ ]:
# TODO (Student Exercise)
for latent_dim in [1, 2, 3]:
    model = Autoencoder(in_dim=3, latent_dim=latent_dim)
    X = make_data(300)
    for _ in range(200):
        X_hat, cache = model.forward(X)
        grads = model.backward(cache)
        model.step(grads, lr=0.05)
    loss = model.mse_loss(X, model.forward(X)[0])
    print('latent_dim', latent_dim, 'loss', round(float(loss), 6))

## Quick Quiz

**Q1.** Why does bottleneck force useful representations?

**Answer:** It compresses information, encouraging structure capture over identity copy.

**Q2.** How are anomalies detected with autoencoders?

**Answer:** Inputs with unusually high reconstruction error are flagged.

**Q3.** Why can overcomplete latent hurt anomaly detection?

**Answer:** Model may reconstruct both normal and abnormal patterns too easily.


## Homework
Implement denoising autoencoder by corrupting input and reconstructing original clean sample.